In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gauravchhajed/idid-dataset/ReadMe -V1.2 for IDID (1).pdf
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/labels_v1.2.json
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/3002017949 -V1.2 for IDID.pdf
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/170311 (2)v.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/160673h.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/160731v.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/100288h.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/140064.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/150454.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/161516v.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/100023d.JPG

In [2]:
!pip install opencv-python tqdm

In [3]:
import json
import os
import cv2
from tqdm import tqdm

In [4]:
BASE_PATH = "/kaggle/input/datasets/gauravchhajed/idid-dataset"

IMG_PATH = BASE_PATH + "/Train_IDID_V1.2/Train/Images"
JSON_PATH = BASE_PATH + "/Train_IDID_V1.2/Train/labels_v1.2.json"

OUTPUT_PATH = "/kaggle/working/yolo_dataset"

os.makedirs(OUTPUT_PATH + "/images/train", exist_ok=True)
os.makedirs(OUTPUT_PATH + "/labels/train", exist_ok=True)

In [5]:
def get_class_id(obj):
    if "conditions" not in obj:
        return None
    
    cond = obj["conditions"]
    
    # Convert all values to string list
    values = list(cond.values())
    
    # Good
    if any("No issues" in v for v in values):
        return 0
    
    # Broken
    if any("Broken" in v for v in values):
        return 1
    
    # Flashover
    if any("Flashover" in v for v in values):
        return 2
    
    return None

In [6]:
import shutil

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

for item in tqdm(data):
    filename = item["filename"]
    img_file = os.path.join(IMG_PATH, filename)
    
    if not os.path.exists(img_file):
        continue
    
    img = cv2.imread(img_file)
    if img is None:
        continue
    
    h, w, _ = img.shape
    
    label_lines = []
    objects = item["Labels"]["objects"]
    
    for obj in objects:
        
        # Skip insulator string
        if obj.get("string", 0) == 1:
            continue
        
        class_id = get_class_id(obj)
        if class_id is None:
            continue
        
        x, y, bw, bh = obj["bbox"]
        
        # Clamp bbox
        x = max(0, x)
        y = max(0, y)
        bw = min(bw, w - x)
        bh = min(bh, h - y)
        
        # Convert to YOLO format
        x_center = (x + bw / 2) / w
        y_center = (y + bh / 2) / h
        bw /= w
        bh /= h
        
        label_lines.append(f"{class_id} {x_center} {y_center} {bw} {bh}")
    
    # Save label
    label_name = os.path.splitext(filename)[0] + ".txt"
    label_file = os.path.join(OUTPUT_PATH, "labels/train", label_name)
    
    with open(label_file, "w") as f:
        f.write("\n".join(label_lines))
    
    # Copy image
    shutil.copy(img_file, os.path.join(OUTPUT_PATH, "images/train"))

100%|██████████| 1600/1600 [02:26<00:00, 10.92it/s]


In [7]:
from collections import Counter

class_counts = Counter()

labels_path = "/kaggle/working/yolo_dataset/labels/train"

for file in os.listdir(labels_path):
    with open(os.path.join(labels_path, file)) as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts[cls] += 1

print(class_counts)

Counter({0: 13060, 2: 2564, 1: 1180})


In [8]:
empty_labels = 0

for file in os.listdir(labels_path):
    path = os.path.join(labels_path, file)
    
    with open(path) as f:
        if len(f.readlines()) == 0:
            empty_labels += 1

print("Empty labels:", empty_labels)

Empty labels: 4


In [9]:
import os
import random
import shutil

random.seed(42)

BASE = "/kaggle/working/yolo_dataset"

train_img_path = f"{BASE}/images/train"
train_label_path = f"{BASE}/labels/train"

val_img_path = f"{BASE}/images/val"
val_label_path = f"{BASE}/labels/val"

# Create val folders
os.makedirs(val_img_path, exist_ok=True)
os.makedirs(val_label_path, exist_ok=True)

# Get all images
images = os.listdir(train_img_path)
random.shuffle(images)

split = int(0.8 * len(images))

train_imgs = images[:split]
val_imgs = images[split:]

# 🔥 MOVE validation data (NOT COPY)
for img in val_imgs:
    src_img = os.path.join(train_img_path, img)
    dst_img = os.path.join(val_img_path, img)

    shutil.move(src_img, dst_img)

    label = os.path.splitext(img)[0] + ".txt"
    
    src_label = os.path.join(train_label_path, label)
    dst_label = os.path.join(val_label_path, label)

    if os.path.exists(src_label):
        shutil.move(src_label, dst_label)

print("Train images:", len(os.listdir(train_img_path)))
print("Val images:", len(os.listdir(val_img_path)))

Train images: 1280
Val images: 320


In [10]:
print("Train images:", len(os.listdir(f"{BASE}/images/train")))
print("Val images:", len(os.listdir(f"{BASE}/images/val")))

Train images: 1280
Val images: 320


In [11]:
data_yaml = f"""
path: /kaggle/working/yolo_dataset

train: images/train
val: images/val

names:
  0: good
  1: broken
  2: flashover
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)

In [12]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.0 MB/s eta 0:00:00


In [13]:
import shutil
import os
import uuid

labels_path = "/kaggle/working/yolo_dataset/labels/train"
images_path = "/kaggle/working/yolo_dataset/images/train"

image_files = os.listdir(images_path)
image_map = {os.path.splitext(img)[0]: img for img in image_files}

for file in os.listdir(labels_path):
    
    with open(os.path.join(labels_path, file)) as f:
        lines = f.readlines()
    
    for line in lines:
        if line.startswith("1 "):  # Broken class
            
            base_name = os.path.splitext(file)[0]
            
            if base_name not in image_map:
                continue
            
            img_name = image_map[base_name]
            
            for _ in range(2):  # 🔥 duplicate twice
                
                unique_id = str(uuid.uuid4())[:6]
                
                new_img_name = f"copy_{unique_id}_{img_name}"
                new_label_name = f"copy_{unique_id}_{file}"
                
                shutil.copy(
                    os.path.join(images_path, img_name),
                    os.path.join(images_path, new_img_name)
                )
                
                shutil.copy(
                    os.path.join(labels_path, file),
                    os.path.join(labels_path, new_label_name)
                )
            
            break

In [14]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # Teacher model

model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    
    # 🔥 performance boost
    cache=True,
    pretrained=True,
    
    # 🔥 augmentation (important for imbalance)
    augment=True,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    close_mosaic=10,
    
    # training stability
    patience=10,
    verbose=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a13e1627cb0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [15]:
!cp /kaggle/working/runs/detect/train/weights/best.pt /kaggle/working/